# Fase 3.9 ? TGIF SD2-fr pareado no Colab

Execute 1?5 para configurar. Execute 6 para congelar os pares de treino TGIF; 7 copia dados internos para /content; 8 e o smoke test (GPU, hashes, checkpoint e forward, sem treino). **A celula 9 inicia ou retoma o treino somente quando executada manualmente.** A 10 le resultados internos. Apos o treino, 11?14 fazem a comparacao externa Mirage com pesos e limiar congelados. Nao use 'Run All'. O Mirage ja foi observado e nao seleciona parametros.


In [ ]:
# 1. GPU e dependências do runtime.
import torch, sys, subprocess
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab antes de continuar'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cryptography', 'huggingface_hub', 'pyarrow', 'scikit-learn'])
print('Dependências disponíveis.', flush=True)

In [ ]:
# 2. Drive e pastas novas; não altera experimentos antigos.
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for name in ('credenciais', 'dados_fase3_9', 'modelos', 'resultados'):
    (ROOT / name).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT, '| livre reportado:', round(shutil.disk_usage(ROOT).free / 1e9, 2), 'GB')
print('/content livre:', round(shutil.disk_usage('/content').free / 1e9, 2), 'GB')

In [ ]:
# 3. Desbloqueio seguro da credencial GitHub existente (formatos antigo e novo).
import base64, getpass, json
from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
credential_path = ROOT / 'credenciais/github.enc'
credential = json.loads(credential_path.read_text(encoding='utf-8'))
if not isinstance(credential, dict) or 'salt' not in credential or ('token' in credential) == ('ciphertext' in credential):
    raise ValueError('Formato da credencial GitHub não reconhecido')
if 'ciphertext' in credential:
    if credential.get('version') != 1: raise ValueError('Versão de credencial não suportada')
    iterations, encrypted = int(credential['iterations']), credential['ciphertext']
else:
    iterations, encrypted = 600000, credential['token']
if iterations < 600000: raise ValueError('KDF insuficiente')
password = getpass.getpass('Senha de desbloqueio do GitHub: ')
key = base64.urlsafe_b64encode(PBKDF2HMAC(algorithm=hashes.SHA256(), length=32, salt=base64.b64decode(credential['salt']), iterations=iterations).derive(password.encode()))
try:
    token = Fernet(key).decrypt(encrypted.encode()).decode()
except InvalidToken:
    raise ValueError('Senha incorreta ou credencial corrompida') from None
finally:
    del password, key
print('Credencial desbloqueada nesta sessão.')

In [ ]:
# 4. Snapshot privado imutável da branch, somente leitura; sem token em URL ou Git.
import io, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
API = 'https://api.github.com/repos/vicemnovo/TCC_Final'
BRANCH = 'fase3-ajuste-fino'
def github_read(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase39-Colab'})
    try:
        with urllib.request.urlopen(req, timeout=120) as response:
            chunks, size, notice = [], 0, time.monotonic()
            while block := response.read(1024 * 1024):
                chunks.append(block); size += len(block)
                if time.monotonic() - notice >= 30:
                    print(f'Download do código: {size / 1048576:.1f} MiB', flush=True); notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code}; confira token, repositório e branch') from None
try:
    commit = json.loads(github_read(f'{API}/branches/{BRANCH}'))['commit']['sha']
    assert re.fullmatch(r'[0-9a-f]{40}', commit)
    CODE_ROOT = Path('/content/tcc_codigo') / commit
    required = ('experimentos/fase3_8_aigi_piloto.py', 'experimentos/fase3_8_aigi_dados.py', 'experimentos/fase3_9_tgif_piloto.py', 'experimentos/fase3_9_tgif_dados.py', 'experimentos/fase3_9_tgif_finetune.py', 'experimentos/avaliar_fase3_9_mirage.py', 'experimentos/fase3_5_dados.py', 'experimentos/fase3_7_dados.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
    if not CODE_ROOT.exists():
        print('Baixando snapshot do GitHub...', flush=True)
        archive = github_read(f'{API}/zipball/{commit}')
        CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(prefix='tcc39_', dir=CODE_ROOT.parent) as temporary:
            staged = Path(temporary)
            with zipfile.ZipFile(io.BytesIO(archive)) as bundle:
                for item in bundle.infolist():
                    parts = Path(item.filename).parts
                    if item.is_dir() or len(parts) < 2: continue
                    relative = Path(*parts[1:])
                    if relative.is_absolute() or '..' in relative.parts: raise ValueError('Snapshot contém caminho inseguro')
                    target = staged / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with bundle.open(item) as source, target.open('wb') as output: shutil.copyfileobj(source, output)
            if not all((staged / name).is_file() for name in required):
                raise FileNotFoundError('Publique primeiro os arquivos da Fase 3.9 na branch')
            staged.rename(CODE_ROOT)
    assert all((CODE_ROOT / name).is_file() for name in required)
finally:
    del token
print('Código sincronizado:', commit)

In [ ]:
# 5. Comandos e progresso periódico; nenhum treino é iniciado aqui.
import os, queue, subprocess, sys, threading, time
DATA = CODE_ROOT / 'experimentos/fase3_9_tgif_dados.py'
TRAIN = CODE_ROOT / 'experimentos/fase3_9_tgif_finetune.py'
for path in (DATA, TRAIN):
    if not path.is_file(): raise FileNotFoundError(path)
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
def run_live(label, command, *, child_env=None):
    print(label + ': iniciando', flush=True)
    process = subprocess.Popen(command, env=ENV if child_env is None else child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = queue.Queue()
    def reader():
        for line in process.stdout: lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    begin = last = time.monotonic()
    while True:
        try:
            line = lines.get(timeout=5)
            if line is None: break
            print(line, end='', flush=True)
            last = time.monotonic()
        except queue.Empty:
            if time.monotonic() - last >= 30:
                print(f'{label}: ativo há {(time.monotonic()-begin)/60:.1f} min; sem nova saída', flush=True)
                last = time.monotonic()
    code = process.wait()
    if code: raise RuntimeError(f'{label} falhou (código {code}); veja o log acima')
    print(f'{label}: concluído em {(time.monotonic()-begin)/60:.1f} min', flush=True)
print('Código:', CODE_ROOT)

In [ ]:
# 6. PREPARO MANUAL: baixa arquivos training TGIF completos para /content; fixa 768 pares no Drive.
# Retome esta celula se a conexao cair. Nao acessa TGIF validation/testing nem benchmark externo.
run_live('preparo TGIF 3.9', [sys.executable, '-u', str(DATA), 'prepare', '--root', str(ROOT)])


In [ ]:
# 7. COPIA LOCAL: execute em cada sessao antes do smoke ou treino.
OLD_DATA = CODE_ROOT / 'experimentos/fase3_5_dados.py'
CF_DATA = CODE_ROOT / 'experimentos/fase3_7_dados.py'
AIGI_DATA = CODE_ROOT / 'experimentos/fase3_8_aigi_dados.py'
for label, script in [('CF 3.5', OLD_DATA), ('CF 3.7', CF_DATA), ('AIGI 3.8', AIGI_DATA), ('TGIF 3.9', DATA)]:
    run_live('copia local '+label, [sys.executable, '-u', str(script), 'stage', '--root', str(ROOT)])


In [ ]:
# 8. SMOKE MANUAL: confere divisoes, hashes, GPU, checkpoint inicial, forward e escrita. Nao treina.
run_live('smoke 3.9', [sys.executable, '-u', str(TRAIN), 'smoke'])


In [ ]:
# 9. TREINO MANUAL: execute somente apos Smoke OK; auto inicia ou retoma last.pt.
required = ('run_live', 'TRAIN', 'ROOT', 'CODE_ROOT', 'ENV', 'sys', 'Path')
if any(name not in globals() for name in required):
    raise RuntimeError('Execute as celulas 1 a 7 nesta sessao antes de iniciar o treino')
local_manifests = [Path('/content/fase3_5_novos_v1/manifest.json'), Path('/content/fase3_7_cfsmall_expansao_v1/manifest.json'), Path('/content/aigi_pares_256_v1/manifest.json'), Path('/content/tgif_sd2fr_768_v1/manifest.json')]
if not all(path.is_file() for path in local_manifests):
    raise FileNotFoundError('Faltam dados locais; execute a celula 7')
run_live('ajuste fino 3.9', [sys.executable, '-u', str(TRAIN), 'auto'])


In [ ]:
# 10. LEITURA INTERNA: checkpoint da melhor epoca e calibracao interna.
run_dir = ROOT / 'modelos/fase3_9_tgif_sd2fr_v1'
for name in ('experiment_config.json','split_manifest.json','history.csv','last.pt','best.pt','phase3_final.pt','internal_metrics.json','training_complete.json'):
    print(name, 'presente' if (run_dir/name).is_file() else 'AUSENTE')
metrics = run_dir/'internal_metrics.json'
print(json.dumps(json.loads(metrics.read_text(encoding='utf-8')),indent=2) if metrics.is_file() else 'Treino interno ainda nao concluido')


## Comparacao externa Mirage

Execute as celulas 11?14 somente depois de concluir o treino interno. O congelamento registra checkpoint, temperatura e limiar antes da inferencia. O Mirage foi observado em fases anteriores e serve apenas para comparacao descritiva; suas respostas nao devem orientar o proximo ajuste.


In [ ]:
# 11. MIRAGE: copia verificada das mesmas 2.000 imagens para /content.
MIRAGE_DATA = CODE_ROOT / 'experimentos/benchmark_mirage_2000.py'
run_live('copia local Mirage', [sys.executable, '-u', str(MIRAGE_DATA), 'stage', '--root', str(ROOT)])


In [ ]:
# 12. CONGELAMENTO antes da avaliacao.
EVAL = CODE_ROOT / 'experimentos/avaliar_fase3_9_mirage.py'
run_live('congelamento 3.9', [sys.executable, '-u', str(EVAL), 'freeze', '--root', str(ROOT)])


In [ ]:
# 13. AVALIACAO EXTERNA COMPARATIVA; nao calibra nem treina.
run_live('avaliacao Mirage 3.9', [sys.executable, '-u', str(EVAL), 'evaluate', '--root', str(ROOT)])


In [ ]:
# 14. LEITURA do resultado preservado.
result = ROOT / 'resultados/benchmark_externo_mirage_2000_v1/fase3_9_tgif_sd2fr_v1/metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')),indent=2) if result.is_file() else 'Avaliacao comparativa ainda nao executada')
